In [1]:
!pip install -qU simalign


In [2]:
from simalign import SentenceAligner

# 初始化 SimAlign 模型
# 使用 "bert" 作為預訓練模型，也可以改成其他支持的變壓器模型
aligner = SentenceAligner(model="bert", token_type="bpe", matching_methods="mai")

# 定義中英文句子
sentence_zh = "人工智能是未來科技的核心技術。"
sentence_en = "Artificial intelligence is the core technology of future science."

# 執行對齊
alignments = aligner.get_word_aligns(sentence_zh, sentence_en)

# 輸出結果
print("詞對齊:")
print(alignments)


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2024-09-11 09:06:41,591 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
INFO:simalign.simalign:Initialized the EmbeddingLoader with model: bert-base-multilingual-cased


詞對齊:
{'mwmf': [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8)], 'inter': [(0, 0), (0, 1), (0, 2), (0, 4), (0, 5), (0, 7), (0, 8)], 'itermax': [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 7), (0, 8)]}


In [3]:
from simalign import SentenceAligner

# 定義中英文句子
chinese_sentence = "人工智能是未來科技的核心技術。"
english_sentence = "Artificial intelligence is the core technology of future science."

# 初始化模型
# 'bert' 模型，fine-tuned 訓練適用於更多語言對齊
aligner = SentenceAligner(model="bert", token_type="bpe", matching_methods="mai")


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2024-09-11 09:06:45,235 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
INFO:simalign.simalign:Initialized the EmbeddingLoader with model: bert-base-multilingual-cased


In [4]:
# 執行對齊，返回對齊結果
alignment = aligner.get_word_aligns(chinese_sentence.split(), english_sentence.split())

# 輸出結果
print("詞對齊:")
print(alignment)


詞對齊:
{'mwmf': [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8)], 'inter': [(0, 0), (0, 1), (0, 2), (0, 4), (0, 5), (0, 7), (0, 8)], 'itermax': [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 7), (0, 8)]}


In [5]:
for method, aligns in alignment.items():
    print(f"# --- 對齊方法: {method} --- #")
    for pair in aligns:
        print(f"中文詞: {chinese_sentence[pair[0]]} -> 英文詞: {english_sentence[pair[1]]}")


# --- 對齊方法: mwmf --- #
中文詞: 人 -> 英文詞: A
中文詞: 人 -> 英文詞: r
中文詞: 人 -> 英文詞: t
中文詞: 人 -> 英文詞: i
中文詞: 人 -> 英文詞: f
中文詞: 人 -> 英文詞: i
中文詞: 人 -> 英文詞: c
中文詞: 人 -> 英文詞: i
中文詞: 人 -> 英文詞: a
# --- 對齊方法: inter --- #
中文詞: 人 -> 英文詞: A
中文詞: 人 -> 英文詞: r
中文詞: 人 -> 英文詞: t
中文詞: 人 -> 英文詞: f
中文詞: 人 -> 英文詞: i
中文詞: 人 -> 英文詞: i
中文詞: 人 -> 英文詞: a
# --- 對齊方法: itermax --- #
中文詞: 人 -> 英文詞: A
中文詞: 人 -> 英文詞: r
中文詞: 人 -> 英文詞: t
中文詞: 人 -> 英文詞: i
中文詞: 人 -> 英文詞: f
中文詞: 人 -> 英文詞: i
中文詞: 人 -> 英文詞: i
中文詞: 人 -> 英文詞: a


In [6]:
# 定義專有名詞對應字典
special_terms = {
    "人工智能": "Artificial Intelligence",
    "核心技術": "core technology"
}

def correct_special_terms(sentence, alignments, special_terms):
    # 取得詞對齊
    src_tokens = alignments['src_tokens']
    tgt_tokens = alignments['tgt_tokens']

    # 進行專有名詞的替換
    for src_idx, tgt_idx in alignments['src_idx2tgt_idx'].items():
        zh_term = src_tokens[src_idx]
        if zh_term in special_terms:
            en_term = special_terms[zh_term]
            tgt_tokens[tgt_idx] = en_term

    # 將修正後的英文詞語合併回完整句子
    corrected_sentence = " ".join(tgt_tokens)
    return corrected_sentence

# 應用專有名詞修正
corrected_sentence = correct_special_terms(sentence_en, alignments, special_terms)
print("修正後的英文句子:")
print(corrected_sentence)


KeyError: 'src_tokens'

## jieba

In [7]:
import jieba
from simalign import SentenceAligner

# 定義中英文句子
chinese_sentence = "我正在學習如何使用對齊技術處理專有名詞。"
english_sentence = "I am learning how to use alignment techniques to handle proper nouns."

# 使用 jieba 進行中文分詞
chinese_words = list(jieba.cut(chinese_sentence))

# 初始化模型
aligner = SentenceAligner(model="bert", token_type="bpe", matching_methods="mai")

# 執行對齊
alignment = aligner.get_word_aligns(chinese_words, english_sentence.split())

# 輸出結果
for method, aligns in alignment.items():
    print(f"--- 對齊方法: {method} ---")
    for pair in aligns:
        print(f"中文詞: {chinese_words[pair[0]]} -> 英文詞: {english_sentence.split()[pair[1]]}")


Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
DEBUG:jieba:Loading model from cache /tmp/jieba.cache
Loading model cost 2.192 seconds.
DEBUG:jieba:Loading model cost 2.192 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2024-09-11 09:06:57,035 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
INFO:simalign.simalign:Initialized the EmbeddingLoader with model: bert-base-multilingual-cased


--- 對齊方法: mwmf ---
中文詞: 我 -> 英文詞: I
中文詞: 正在 -> 英文詞: am
中文詞: 學習 -> 英文詞: learning
中文詞: 如何 -> 英文詞: how
中文詞: 使用 -> 英文詞: to
中文詞: 使用 -> 英文詞: use
中文詞: 對齊 -> 英文詞: alignment
中文詞: 技術 -> 英文詞: techniques
中文詞: 技術 -> 英文詞: to
中文詞: 處理專 -> 英文詞: handle
中文詞: 有名 -> 英文詞: proper
中文詞: 有名 -> 英文詞: nouns.
中文詞: 詞 -> 英文詞: nouns.
中文詞: 。 -> 英文詞: nouns.
--- 對齊方法: inter ---
中文詞: 我 -> 英文詞: I
中文詞: 正在 -> 英文詞: am
中文詞: 學習 -> 英文詞: learning
中文詞: 如何 -> 英文詞: how
中文詞: 使用 -> 英文詞: use
中文詞: 技術 -> 英文詞: techniques
中文詞: 處理專 -> 英文詞: handle
中文詞: 詞 -> 英文詞: nouns.
中文詞: 。 -> 英文詞: nouns.
--- 對齊方法: itermax ---
中文詞: 我 -> 英文詞: I
中文詞: 正在 -> 英文詞: am
中文詞: 學習 -> 英文詞: learning
中文詞: 如何 -> 英文詞: how
中文詞: 使用 -> 英文詞: to
中文詞: 使用 -> 英文詞: use
中文詞: 技術 -> 英文詞: alignment
中文詞: 技術 -> 英文詞: techniques
中文詞: 處理專 -> 英文詞: handle
中文詞: 有名 -> 英文詞: proper
中文詞: 有名 -> 英文詞: nouns.
中文詞: 詞 -> 英文詞: nouns.
中文詞: 。 -> 英文詞: nouns.


## spacy


In [5]:
!pip install -qU spacy
!python -m spacy download zh_core_web_sm
!python -m spacy download zh_core_web_trf


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 MB 12.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('zh_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.1/415.1 MB 3.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('zh_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [9]:
!python -m spacy download zh_core_web_trf

  Using cached https://github.com/explosion/spacy-models/releases/download/zh_core_web_trf-3.7.2/zh_core_web_trf-3.7.2-py3-none-any.whl (415.1 MB)
✔ Download and installation successful
You can now load the package via spacy.load('zh_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [10]:
import spacy
from simalign import SentenceAligner

# Step 1: 載入 spacy 中文模型
nlp = spacy.load("zh_core_web_sm")

# Step 2: 定義中文和英文句子
chinese_sentence = "臺灣大學的學生來參加這次比賽。"
english_sentence = "The students from National Taiwan University are here to participate in this competition."

# Step 3: 使用 spacy 進行中文分詞
doc = nlp(chinese_sentence)
chinese_words = [token.text for token in doc]

# Step 4: 初始化 SimAlign 模型
aligner = SentenceAligner(model="bert", token_type="bpe", matching_methods="mai")

# Step 5: 執行詞對齊
alignment = aligner.get_word_aligns(chinese_words, english_sentence.split())

# Step 6: 輸出原句與翻譯句
print("原本的句子: " + chinese_sentence)
print("翻譯的句子: " + english_sentence)

# Step 7: 建議調整句子（根據詞對齊）
adjusted_translation = english_sentence.split()  # 複製英文翻譯句子，便於後續調整

# 利用 itermax 的對齊來調整翻譯
for pair in alignment['itermax']:
    # 獲取中文詞和對應的英文詞
    chinese_word = chinese_words[pair[0]]
    english_word = english_sentence.split()[pair[1]]

    # 如果翻譯不準確，這裡可以進行手動調整，或做註解提示
    # 比如根據上下文來調整特定詞彙的翻譯
    # print(f"'{chinese_word}' 對應到 '{english_word}'")
    # 這裡你可以根據對齊情況來調整翻譯，例如：
    # 如果發現 "比賽" 翻譯成 "game"，而不是 "competition"，就可以調整
    # adjusted_translation[pair[1]] = "corrected_term"

# Step 8: 最後組合建議調整後的句子
suggested_sentence = " ".join(adjusted_translation)

# Step 9: 輸出建議調整的句子
print("建議調整的句子: " + suggested_sentence)


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2024-09-11 09:08:51,030 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
INFO:simalign.simalign:Initialized the EmbeddingLoader with model: bert-base-multilingual-cased


原本的句子: 臺灣大學的學生來參加這次比賽。
翻譯的句子: The students from National Taiwan University are here to participate in this competition.
建議調整的句子: The students from National Taiwan University are here to participate in this competition.


## 專有名詞重複的修改

ref: https://chatgpt.com/share/e/92cf31e3-eee4-4ea7-864c-34002137b0b6

In [14]:
import spacy
from simalign import SentenceAligner

# Step 1: 載入 spacy 中文模型
nlp = spacy.load("zh_core_web_sm")

# Step 2: 定義中文和英文句子
chinese_sentence = "高雄的六合夜市是美食愛好者的天堂。"
english_sentence = "Kaohsiung's Liuhe Night Market is a paradise for gourmet enthusiasts."

# Step 3: 使用 spacy 進行中文分詞
doc = nlp(chinese_sentence)
chinese_words = [token.text for token in doc]

# Step 4: 初始化 SimAlign 模型
aligner = SentenceAligner(model="bert", token_type="bpe", matching_methods="mai")

# Step 5: 執行詞對齊
alignment = aligner.get_word_aligns(chinese_words, english_sentence.split())

# Step 6: 輸出原句與翻譯句
print("原本的句子: " + chinese_sentence)
print("翻譯的句子: " + english_sentence)

# Step 7: 建議調整句子，保留專有名詞（根據詞對齊）
# 假設專有名詞是 "高雄" 和 "六合夜市"，並且在句子中我們希望保留它們
proper_nouns = set(["高雄", "六合夜市"])

# 準備調整的翻譯，並建立一個集合來追踪已經替換過的中文詞和索引
adjusted_translation = english_sentence.split()
replaced_words = set()  # 用來跟踪已經替換的中文專有名詞
replaced_indices = set()  # 用來跟踪已經替換的英文索引

# 進行對齊並保留專有名詞
for pair in alignment['itermax']:
    chinese_word = chinese_words[pair[0]]
    english_word_index = pair[1]

    # 如果這個中文詞是專有名詞，並且尚未替換過
    if chinese_word in proper_nouns and chinese_word not in replaced_words and english_word_index not in replaced_indices:
        adjusted_translation[english_word_index] = chinese_word
        replaced_words.add(chinese_word)  # 標記此中文詞已經替換
        replaced_indices.add(english_word_index)  # 標記此索引已經替換

# Step 8: 組合建議調整後的句子
suggested_sentence = " ".join(adjusted_translation)

# Step 9: 輸出建議調整的句子
print("建議調整的句子: " + suggested_sentence)


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2024-09-11 09:21:13,557 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
INFO:simalign.simalign:Initialized the EmbeddingLoader with model: bert-base-multilingual-cased


原本的句子: 高雄的六合夜市是美食愛好者的天堂。
翻譯的句子: Kaohsiung's Liuhe Night Market is a paradise for gourmet enthusiasts.
建議調整的句子: 高雄 六合夜市 Night Market is a paradise for gourmet enthusiasts.


## 30 個繁體中文句子

* Generated by Chatgpt-4o.
* Ref with [a messy dialogue](https://chatgpt.com/share/d5b671c8-432d-4465-a8bc-5255c73b3ee6).

In [1]:
test_sentence = {
  "sentences": [
    "張三在台北101前面拍了很多照片。",
    "李四昨天在台中火車站迷路了。",
    "王五參加了鴻海公司的年度大會。",
    "華為手機在全球市場上的銷量持續增長。",
    "張三開了一間叫做「香格里拉」的餐廳。",
    "小明帶著他的iPhone去了華山藝文中心。",
    "你知道Apple的總部在哪裡嗎？",
    "昨天在星巴克遇見了老朋友林小明。",
    "上海是中國的經濟中心。",
    "Google正在開發一個新的AI模型。",
    "馬雲創辦的阿里巴巴是中國最大的電商平台之一。",
    "珠穆朗瑪峰是世界最高的山。",
    "高雄的六合夜市是美食愛好者的天堂。",
    "臺灣大學的學生來參加這次比賽。",
    "蘋果電腦的發明改變了整個科技產業。",
    "中國移動的用戶數量每年都在增加。",
    "香港的金融市場非常活躍。",
    "IBM推出了一款全新的量子計算機。",
    "Microsoft的Windows系統是全球最常用的操作系統之一。",
    "劉德華出演了很多經典的電影。",
    "台灣的玉山是當地最高的山峰。",
    "世界衛生組織在日內瓦總部召開會議。",
    "任天堂的Switch遊戲機在全球大受歡迎。",
    "中央電視台播放了關於新冠疫情的最新報導。",
    "奧林匹克運動會將在東京舉行。",
    "特斯拉的自動駕駛技術引發了廣泛關注。",
    "小紅書是一個非常受歡迎的社交平台。",
    "Facebook的隱私政策再次引發爭議。",
    "日本的富士山每年都吸引大量遊客。",
    "李小龍的武術精神影響了全世界。"
  ]
}

In [2]:
proper_nouns = set([
    "張三", "台北101", "李四", "台中火車站", "王五", "鴻海公司", "華為",
    "香格里拉", "小明", "iPhone", "華山藝文中心", "Apple", "星巴克", "林小明",
    "上海", "中國", "Google", "馬雲", "阿里巴巴", "珠穆朗瑪峰", "高雄",
    "六合夜市", "臺灣大學", "蘋果電腦", "中國移動", "香港", "IBM",
    "Microsoft", "Windows", "劉德華", "玉山", "世界衛生組織", "日內瓦",
    "任天堂", "Switch", "中央電視台", "新冠疫情", "奧林匹克運動會", "東京",
    "特斯拉", "小紅書", "Facebook", "富士山", "李小龍"
])

In [3]:
# 用 google 套件翻譯
!pip install -q googletrans==4.0.0-rc1

In [7]:
# 使用 spacy 進行中文分詞
import spacy
nlp = spacy.load("zh_core_web_sm")

# 翻譯器
import googletrans
from googletrans import Translator

translator = Translator()




/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2024-09-11 09:16:11,550 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
INFO:simalign.simalign:Initialized the EmbeddingLoader with model: bert-base-multilingual-cased


In [11]:
from simalign import SentenceAligner
src_sent = []
trans_res = []
trans_adj_res = []

for source_sentence in test_sentence["sentences"]:
  src_sent.append(source_sentence)
  current_translation = translator.translate(source_sentence, src = "zh-TW", dest='en')
  trans_res.append(current_translation.text)


  #  作分詞
  doc = nlp(source_sentence)
  chinese_words = [token.text for token in doc]

  #  初始化 SimAlign 模型
  aligner = SentenceAligner(model="bert", token_type="bpe", matching_methods="mai")
  #  執行詞對齊
  alignment = aligner.get_word_aligns(chinese_words, current_translation.text.split())


  # 準備調整的翻譯，並建立一個集合來追踪已經替換過的索引
  adjusted_translation = current_translation.text.split()  # 複製英文翻譯句子，便於後續調整
  replaced_words = set()  # 用來跟踪已經替換的中文專有名詞
  replaced_indices = set()

  # 利用 itermax 的對齊來調整翻譯
  for pair in alignment['itermax']:
    # 獲取中文詞和對應的英文詞
    chinese_word = chinese_words[pair[0]]
    english_word_index = pair[1]
    english_word = current_translation.text.split()[pair[1]]

    # 如果這個中文詞是專有名詞，並且尚未在該索引處替換過
    if chinese_word in proper_nouns and chinese_word not in replaced_words and english_word_index not in replaced_indices:
      adjusted_translation[english_word_index] = chinese_word
      replaced_words.add(chinese_word)  # 標記此中文詞已經替換
      replaced_indices.add(english_word_index)  # 標記此索引已經替換

        # # 如果這個中文詞是專有名詞，我們保留它的中文形式到翻譯中
        # if chinese_word in proper_nouns:
        #   adjusted_translation[english_word_index] = chinese_word

    # 最後組合建議調整後的句子
    suggested_sentence = " ".join(adjusted_translation)
    trans_adj_res.append(suggested_sentence)



/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2024-09-11 09:18:35,430 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
INFO:simalign.simalign:Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn

In [12]:
import pandas as pd

df = pd.DataFrame(list(zip(src_sent, trans_res, trans_adj_res)),
               columns =['原句', '初步翻譯', '調整後翻譯'])

In [13]:
df.head(30)

,原句,初步翻譯,調整後翻譯
0,張三在台北101前面拍了很多照片。,Zhang San took a lot of photos in front of Tai...,張三 San took a lot of photos in front of Taipei...
1,李四昨天在台中火車站迷路了。,Li Si was lost at Taichung Railway Station yes...,張三 San took a lot of photos in front of Taipei...
2,王五參加了鴻海公司的年度大會。,Wang Wu participated in the annual conference ...,張三 San took a lot of photos in front of Taipei...
3,華為手機在全球市場上的銷量持續增長。,The sales of Huawei mobile phones in the globa...,張三 San took a lot of photos in front of Taipei...
4,張三開了一間叫做「香格里拉」的餐廳。,"Zhang San opened a restaurant called ""Shangri ...",張三 San took a lot of photos in front of Taipei...
5,小明帶著他的iPhone去了華山藝文中心。,Xiaoming took his iPhone to Huashan Art and Cu...,張三 San took a lot of photos in front of Taipei...
6,你知道Apple的總部在哪裡嗎？,Do you know where the headquarters of Apple is?,張三 San took a lot of photos in front of Taipei...
7,昨天在星巴克遇見了老朋友林小明。,I met an old friend Lin Xiaoming in Starbucks ...,張三 San took a lot of photos in front of Taipei...
8,上海是中國的經濟中心。,Shanghai is the economic center of China.,張三 San took a lot of photos in front of Taipei...
9,Google正在開發一個新的AI模型。,Google is developing a new AI model.,張三 San took a lot of photos in front of Taipei...
